In [32]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

train = pd.read_csv("/kaggle/input/datasets/dormadum/gridlockdataset/train.csv")         
test = pd.read_csv("/kaggle/input/datasets/dormadum/gridlockdataset/test.csv")



In [33]:
train_copy = train.copy()
train_copy['geohash'] = train_copy['geohash'].astype(str)

# 2. Check if ALL strings start with 'gp'
all_start_with_gp = train_copy['geohash'].str.startswith('qp').all()
print(f"all geohashes start with 'qp'?: {all_start_with_gp}\n")

# 3. If they vary, count how many DO NOT start with 'gp'
# Create a boolean mask where True means it DOES NOT start with 'gp'
does_not_start_with_gp = ~train_copy['geohash'].str.startswith('qp')
varying_count = does_not_start_with_gp.sum()

print(f"Number of geohashes that do NOT start with 'qp': {varying_count}")

all geohashes start with 'qp'?: True

Number of geohashes that do NOT start with 'qp': 0


In [34]:
train_hashes = set(train['geohash'].unique())
test_hashes = set(test['geohash'].unique())

print(f"Unique hashes in Train: {len(train_hashes)}")
print(f"Unique hashes in Test: {len(test_hashes)}")
print(f"New hashes in Test set: {len(test_hashes - train_hashes)}")

Unique hashes in Train: 1249
Unique hashes in Test: 1190
New hashes in Test set: 10


In [35]:
print(train['timestamp'].unique()[:15])
print(f"Total unique timestamps in a day: {train[train['day']==48]['timestamp'].nunique()}")

['0:0' '0:15' '0:30' '0:45' '1:0' '1:15' '1:30' '1:45' '2:0' '2:15' '2:30'
 '2:45' '3:0' '3:15' '3:30']
Total unique timestamps in a day: 96


In [36]:
for col in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
    print(f"\n--- {col} Distribution (Train) ---")
    print(train[col].value_counts(dropna=False))
    print(f"--- {col} Distribution (Test) ---")
    print(test[col].value_counts(dropna=False))


--- RoadType Distribution (Train) ---
RoadType
Residential    69230
Street          3909
Highway         3560
NaN              600
Name: count, dtype: int64
--- RoadType Distribution (Test) ---
RoadType
Residential    33775
Highway         4272
Street          3407
NaN              324
Name: count, dtype: int64

--- Weather Distribution (Train) ---
Weather
Sunny    27717
Rainy    20824
Foggy    20243
Snowy     7718
NaN        797
Name: count, dtype: int64
--- Weather Distribution (Test) ---
Weather
Sunny    15078
Foggy    11102
Rainy    11081
Snowy     4086
NaN        431
Name: count, dtype: int64

--- LargeVehicles Distribution (Train) ---
LargeVehicles
Not Allowed    50673
Allowed        26626
Name: count, dtype: int64
--- LargeVehicles Distribution (Test) ---
LargeVehicles
Not Allowed    26178
Allowed        15600
Name: count, dtype: int64

--- Landmarks Distribution (Train) ---
Landmarks
Yes    52042
No     25257
Name: count, dtype: int64
--- Landmarks Distribution (Test) ---
Land

In [37]:
print(train['demand'].describe())
print(f"Skewness: {train['demand'].skew()}")

count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64
Skewness: 3.7285172887316627


In [38]:
print("\nNumber of records per day:")
print(train['day'].value_counts().sort_index())


Number of records per day:
day
48    69427
49     7872
Name: count, dtype: int64


In [39]:

def preprocess_pipeline(df):
    df_out = df.copy()
    
    time_split = df_out['timestamp'].str.split(':', expand=True).astype(int)
    df_out['hour'] = time_split[0]
    df_out['minute'] = time_split[1]
    
    df_out['time_slot'] = (df_out['hour'] * 4) + (df_out['minute'] // 15)
    
    df_out['day_of_week'] = df_out['day'] % 7
    
    # convert string object categories to categorical dtype for LGBM
    cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
    for col in cat_cols:
        df_out[col] = df_out[col].astype('category')
        
    return df_out

train_df = preprocess_pipeline(train)
test_df = preprocess_pipeline(test)

features = [
    'geohash', 'day_of_week', 'time_slot', 'hour', 'minute',
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 
    'Temperature', 'Weather'
]

In [40]:

X_train = train_df[train_df['day'] == 48][features]
y_train = np.log1p(train_df[train_df['day'] == 48]['demand'])  # Log transform to fix skewness

X_val = train_df[train_df['day'] == 49][features]
y_val = train_df[train_df['day'] == 49]['demand']  # Keep raw validation target for pure R2 validation

print(f"📉 Train Shape (Day 48): {X_train.shape} | Validation Shape (Day 49): {X_val.shape}")

📉 Train Shape (Day 48): (69427, 11) | Validation Shape (Day 49): (7872, 11)


In [41]:

print("🚀 Training Phase Baseline LightGBM Model...")
model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, np.log1p(y_val))],
    callbacks=[]
)


val_preds_log = model.predict(X_val)

val_preds = np.expm1(val_preds_log)


🚀 Training Phase Baseline LightGBM Model...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001817 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1461
[LightGBM] [Info] Number of data points in the train set: 69427, number of used features: 10
[LightGBM] [Info] Start training from score 0.082019


In [42]:

local_r2 = r2_score(y_val, val_preds)
local_score = max(0, 100 * local_r2)
print(f"Local Validation R2 Score: {local_r2:.5f}")



Local Validation R2 Score: 0.73968


In [43]:
X_test = test_df[features]
test_preds_log = model.predict(X_test)
test_preds = np.expm1(test_preds_log)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_preds
})

submission.to_csv("baseline_submission.csv", index=False)
print(f"dimensions: {submission.shape}")

dimensions: (41778, 2)


In [44]:
sub_check = pd.read_csv("/kaggle/working/baseline_submission.csv")
print(sub_check.head())
print("Missing values in submission:", sub_check.isna().sum().sum())

   Index    demand
0      0  0.051196
1      1  0.029212
2      2  0.024220
3      3  0.040122
4      4  0.062909
Missing values in submission: 0
